# Baseline Evaluation: Demonstrating Cross-Dataset Generalization Limitation

This notebook demonstrates the cross-dataset generalization problem by:
1. Training a baseline model on FaceForensics++ with basic augmentation
2. Evaluating on FaceForensics++ (intra-dataset) - expect ~95%+ accuracy
3. Evaluating on Celeb-DF and DFDC (cross-dataset) - expect significant drop
4. Documenting performance degradation

In [ ]:
import sys
sys.path.append('..')

import json
import torch
import yaml
from pathlib import Path

from training.train import train_model
from evaluation.evaluate import cross_dataset_evaluation
from utils.visualization import (
    plot_training_history,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_cross_dataset_comparison
)

## 1. Train Baseline Model

Train XceptionNet on FaceForensics++ with **basic augmentation**.

In [ ]:
# Train baseline model with basic augmentation
config_path = '../config/config.yaml'

print("Training baseline model with BASIC augmentation...")
print("This will take some time depending on your hardware.")

# Uncomment to train (may take hours)
# train_model(
#     config_path=config_path,
#     augmentation_type='basic',
#     model_name='xception'
# )

## 2. Visualize Training History

In [ ]:
# Load training history
history_path = '../checkpoints/xception_basic/training_history.json'

if Path(history_path).exists():
    with open(history_path, 'r') as f:
        history = json.load(f)
    
    # Plot training curves
    plot_training_history(
        history,
        save_path='../results/baseline_training_history.png'
    )
else:
    print(f"Training history not found at {history_path}")
    print("Please train the model first.")

## 3. Cross-Dataset Evaluation

Evaluate the baseline model on multiple datasets.

In [ ]:
# Evaluate on all datasets
checkpoint_path = '../checkpoints/xception_basic/best_model.pth'

if Path(checkpoint_path).exists():
    # Run cross-dataset evaluation
    results = cross_dataset_evaluation(
        checkpoint_path=checkpoint_path,
        config_path=config_path,
        output_dir='../results/baseline'
    )
else:
    print(f"Checkpoint not found at {checkpoint_path}")
    print("Please train the model first.")

## 4. Visualize Results

In [ ]:
# Load evaluation results
results_path = '../results/baseline/evaluation_results.json'

if Path(results_path).exists():
    with open(results_path, 'r') as f:
        eval_results = json.load(f)
    
    # Plot cross-dataset comparison
    metrics_dict = eval_results['metrics']
    plot_cross_dataset_comparison(
        metrics_dict,
        save_path='../results/baseline/cross_dataset_comparison.png',
        title='Baseline Model: Cross-Dataset Performance'
    )
else:
    print(f"Evaluation results not found at {results_path}")

## 5. Key Findings

Expected observations:
- **Intra-dataset (FaceForensics++)**: High accuracy (~95%+)
- **Cross-dataset (Celeb-DF, DFDC)**: Significant performance drop (~60-70%)
- **Conclusion**: Baseline model with basic augmentation suffers from poor generalization

In [ ]:
if Path(results_path).exists():
    print("\n" + "="*80)
    print("BASELINE MODEL PERFORMANCE SUMMARY")
    print("="*80)
    
    for dataset_name, metrics in metrics_dict.items():
        print(f"\n{dataset_name}:")
        print(f"  Accuracy: {metrics['accuracy']:.4f}")
        print(f"  F1-Score: {metrics['f1_score']:.4f}")
        print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")